# Proxy Wall Finder — DAP depth (Colab)

Produces `.npz` depth for the web app. **Runtime → T4 GPU** first.

Panoramas are uploaded to Google for this run — use a local GPU for sensitive client sites.
DAP weights are **CC BY-NC 4.0** (non-commercial).


In [ ]:
# 1 · GPU check
import torch
assert torch.cuda.is_available(), 'No GPU — Runtime → Change runtime type → T4 GPU'
print(torch.cuda.get_device_name(0), torch.cuda.get_device_properties(0).total_memory/1e9, 'GB')


In [ ]:
# 2 · Setup: clone DAP + this repo, minimal pip (once)
import os
os.chdir('/content')
!git clone --depth 1 https://github.com/Insta360-Research-Team/DAP.git DAP
!git clone --depth 1 https://github.com/Razee4315/proxy-wall-finder.git proxy-wall-finder
%pip -q install einops torchmetrics opencv-python-headless matplotlib pyyaml numpy
print('setup ok')


In [ ]:
# 3 · Download DAP weights once (model.pth ~1.46 GB, CC BY-NC 4.0)
import os
os.makedirs('/content/DAP/weights', exist_ok=True)
out = '/content/DAP/weights/model.pth'
if not os.path.exists(out) or os.path.getsize(out) < 1e9:
    !wget -q --show-progress -O /content/DAP/weights/model.pth https://huggingface.co/Insta360-Research/DAP-weights/resolve/main/model.pth
size = os.path.getsize(out) / 1e9
assert size > 1.0, size
print(f'weights OK: {size:.2f} GB')


In [ ]:
# 4 · Upload panos
import os
from google.colab import files
os.makedirs('/content/panos', exist_ok=True)
uploaded = files.upload()
for name, data in uploaded.items():
    open(f'/content/panos/{name}', 'wb').write(data)
print(len(uploaded), 'pano(s)')


In [ ]:
# 5 · Generate depth
!python /content/proxy-wall-finder/scripts/generate-depth.py \
  --dap-root /content/DAP \
  --weights  /content/DAP/weights/model.pth \
  --panos    /content/panos \
  --out      /content/depth


In [ ]:
# 6 · Preview
import glob, json, math
import matplotlib.pyplot as plt
import numpy as np
previews = sorted(glob.glob('/content/depth/preview/*.png'))
assert previews, 'No previews'
cols = 2
rows = math.ceil(len(previews)/cols)
fig, axes = plt.subplots(rows, cols, figsize=(14, 4*rows))
flat = list(np.atleast_1d(axes).flat)
for ax, path in zip(flat, previews):
    ax.imshow(plt.imread(path)); ax.set_title(path.split('/')[-1][:-4]); ax.axis('off')
for ax in flat[len(previews):]: ax.axis('off')
plt.show()


In [ ]:
# 7 · Download zip → unzip into proxy-wall-finder/depth/ for the app
from google.colab import files
!cd /content && zip -qr depth-output.zip depth
files.download('/content/depth-output.zip')
